# NLP Task 3 – Scratch Transformer vs Pretrained DistilBERT (Spam/Ham SMS)

**Run order:** Runtime → Change runtime type → **T4 GPU**, then *Run all*. When asked, upload `spam_sms.csv`.

- **Part 1** – Member 1: preprocessing, split, vocabulary, Transformer from scratch, training, evaluation
- **Part 2** – Member 2: fine-tune DistilBERT, evaluate, compare both models

Everything is saved at the end into `project_artifacts.zip` (for Member 3's Streamlit app).

# PART 1 – Scratch Transformer (Member 1)

#1.Libraries

In [ ]:
import pandas as pd
import numpy as np
import math
import re
import json
import os
import time
import random
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import shutil

#2.Choose Device & Seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

#3.Data Reading & Understandnig

In [ ]:
import os
if not os.path.exists("spam_sms.csv"):
    from google.colab import files
    print("Upload spam_sms.csv")
    files.upload()

In [ ]:
df=pd.read_csv("spam_sms.csv")
data=df.copy()
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.value_counts('label')

In [ ]:
print(df["label"].value_counts())

plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x="label",
    order=df["label"].value_counts().index
)

plt.xticks(rotation=45)
plt.title("Class Distribution")
plt.show()

**Class Imbalance problem because more than 50% of data classified as ham messages**

In [ ]:
df.duplicated().sum()

#4.Data Cleaning

In [ ]:
df = df.drop_duplicates()
df.shape

#5.Text Preprocessing

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()
df["clean_message"] = df["message"].apply(clean_text)
df.head()

**Delete Empty Texts**

In [ ]:
df["clean_message"] = df["clean_message"].fillna("")

df = df[df["clean_message"].str.strip() != ""]

df.reset_index(drop=True, inplace=True)

print(df.shape)

#6.Converting Class Labels into Numbers

In [ ]:
label_mapping = {"ham": 0,"spam": 1}
df["label_id"] = df["label"].map(label_mapping)
print(df[["clean_message", "label", "label_id"]].head())

#7.Data after Cleaning


In [ ]:
print("Total messages:", len(df))

print("\nLabel counts:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(df["label"].value_counts(normalize=True) * 100)

#8.Data Splitting : Train/Test/Validation


In [ ]:
train_df, temp_df = train_test_split(df,test_size=0.20,random_state=SEED,stratify=df["label_id"])

val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=SEED,stratify=temp_df["label_id"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

In [ ]:
os.makedirs("shared_data", exist_ok=True)
train_df.to_csv("shared_data/train.csv",index=False)
val_df.to_csv("shared_data/validation.csv",index=False)
test_df.to_csv("shared_data/test.csv",index=False)
print("Shared dataset splits saved successfully!")

#9.Tokenization & Vocabulary

**Tokenization**

In [ ]:
def tokenize_text(text):

    tokens = re.findall(r"[a-z0-9]+|[^\w\s]",str(text).lower())

    return tokens


example = train_df["clean_message"].iloc[0]

print("Original Message:")
print(example)

print("\nTokens:")
print(tokenize_text(example))

**Building Vocabulary**

In [ ]:
def build_vocabulary(texts, max_vocab_size=20000):

    word_counter = Counter()

    for text in texts:

        tokens = tokenize_text(text)

        word_counter.update(tokens)

    # Special tokens
    vocab = {"[PAD]": 0,"[UNK]": 1}

    # Add the most frequent words
    for word, count in word_counter.most_common(max_vocab_size - 2):

        if word not in vocab:

            vocab[word] = len(vocab)

    return vocab


vocab = build_vocabulary(train_df["clean_message"].tolist())

print("Vocabulary size:", len(vocab))

print("\nFirst 20 vocabulary items:")
print(list(vocab.items())[:20])

#10.Convert Words into numbers

In [ ]:
MAX_LEN = 100

def encode_text(text, vocab, max_len=MAX_LEN):

    tokens = tokenize_text(text)

    # Convert tokens to IDs
    token_ids = [
        vocab.get(token, vocab["[UNK]"])
        for token in tokens
    ]

    # Truncate long messages
    token_ids = token_ids[:max_len]

    # Avoid empty sequences
    if len(token_ids) == 0:
        token_ids = [vocab["[UNK]"]]

    # Padding
    padding_length = max_len - len(token_ids)

    token_ids += (
        [vocab["[PAD]"]] * padding_length
    )

    return token_ids


example_ids = encode_text(
    example,
    vocab
)

print("Original message:")
print(example)

print("\nEncoded message:")
print(example_ids)

print("\nEncoded length:")
print(len(example_ids))

#11. Creating Pytorch Dataset and Data Loader

**Creating Data Loaders**

In [ ]:
class SpamDataset(Dataset):

    def __init__(self, dataframe, vocab):

        self.texts = dataframe["clean_message"].tolist()

        self.labels = dataframe["label_id"].tolist()

        self.encoded_texts = [
            encode_text(text, vocab)
            for text in self.texts
        ]

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, index):

        input_ids = torch.tensor(
            self.encoded_texts[index],
            dtype=torch.long
        )

        label = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return input_ids, label

BATCH_SIZE = 32

train_dataset = SpamDataset(train_df, vocab)

val_dataset = SpamDataset(val_df, vocab)

test_dataset = SpamDataset(test_df, vocab)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

#12.Building a Transformer from scratch

**Manual Multi-Head self-Attention**

In [ ]:
class ManualSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout=0.1
    ):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = d_model // num_heads

        # Linear projections
        self.query = nn.Linear(
            d_model, d_model
        )

        self.key = nn.Linear(
            d_model, d_model
        )

        self.value = nn.Linear(
            d_model, d_model
        )

        self.output = nn.Linear(
            d_model, d_model
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x, padding_mask=None):

        batch_size, seq_len, _ = x.shape

        # Step 1: Generate Q, K, V
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Step 2: Split into heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Step 3: Calculate attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        # Step 4: Scale scores
        scores = scores / math.sqrt(
            self.head_dim
        )

        # Step 5: Apply padding mask
        if padding_mask is not None:

            mask = padding_mask[:, None, None, :]

            scores = scores.masked_fill(
                ~mask,
                torch.finfo(scores.dtype).min
            )

        # Step 6: Softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        # Step 7: Multiply by V
        context = torch.matmul(
            attention_weights,
            V
        )

        # Step 8: Combine heads
        context = context.transpose(
            1, 2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # Step 9: Final linear projection
        output = self.output(context)

        return output

**Transformer Encoding Block**

In [ ]:
class TransformerEncoderBlock(nn.Module):

    def __init__(
        self,
        d_model=128,
        num_heads=4,
        ff_dim=256,
        dropout=0.1
    ):

        super().__init__()

        self.attention = ManualSelfAttention(
            d_model,
            num_heads,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(

            nn.Linear(
                d_model,
                ff_dim
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                ff_dim,
                d_model
            )
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x, padding_mask=None):

        # Multi-Head Attention
        attention_output = self.attention(
            x,
            padding_mask
        )

        # Residual Connection + Layer Normalization
        x = self.norm1(
            x + self.dropout(attention_output)
        )

        # Feed Forward Network
        ff_output = self.feed_forward(x)

        # Residual Connection + Layer Normalization
        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x

#13.Building Model

**Scratch Transformer Classifier**

In [ ]:
class ScratchSpamTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        pad_id=0,
        max_len=100,
        d_model=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1,
        num_classes=2
    ):

        super().__init__()

        self.pad_id = pad_id

        # Token Embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_id
        )

        # Positional Embedding
        self.position_embedding = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer Encoder Layers
        self.encoder_layers = nn.ModuleList([

            TransformerEncoderBlock(
                d_model=d_model,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout=dropout
            )

            for _ in range(num_layers)

        ])

        self.dropout = nn.Dropout(dropout)

        # Classification layer
        self.classifier = nn.Linear(
            d_model,
            num_classes
        )


    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        # Position IDs
        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        positions = positions.unsqueeze(0).expand(
            batch_size,
            -1
        )

        # Padding Mask
        padding_mask = (
            input_ids != self.pad_id
        )

        # Embeddings
        token_embeddings = self.token_embedding(
            input_ids
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = (
            token_embeddings +
            position_embeddings
        )

        x = self.dropout(x)

        # Pass through Encoder layers
        for layer in self.encoder_layers:

            x = layer(
                x,
                padding_mask
            )

        # Masked Mean Pooling
        mask = padding_mask.unsqueeze(-1).float()

        pooled = (x * mask).sum(dim=1)

        pooled = pooled / mask.sum(
            dim=1
        ).clamp(min=1.0)

        # Final classification
        logits = self.classifier(pooled)

        return logits

#14.Scratch Transfrmer Training

**Creating Model,Optimizer,Loss**

In [ ]:
# Create model
model = ScratchSpamTransformer(
    vocab_size=len(vocab),
    pad_id=vocab["[PAD]"],
    max_len=MAX_LEN,
    d_model=128,
    num_heads=4,
    num_layers=2,
    ff_dim=256,
    dropout=0.15,
    num_classes=2
)

model = model.to(DEVICE)

# Calculate class weights
class_counts = (
    train_df["label_id"]
    .value_counts()
    .sort_index()
    .values
)

class_weights = len(train_df) / (
    2 * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(DEVICE)

print("Class weights:", class_weights)

# Loss function
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0002,
    weight_decay=0.0001
)

print("\nModel created successfully!")

**Training Loop**

In [ ]:
EPOCHS = 8

best_val_loss = float("inf")

train_losses = []
val_losses = []

os.makedirs("artifacts", exist_ok=True)

start_time = time.time()


for epoch in range(EPOCHS):

    # Training


    model.train()

    total_train_loss = 0

    for input_ids, labels in train_loader:

        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update weights
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = (
        total_train_loss / len(train_loader)
    )


    # Validation


    model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for input_ids, labels in val_loader:

            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(input_ids)

            loss = criterion(outputs, labels)

            total_val_loss += (
                loss.item() * len(labels)
            )

    average_val_loss = (
        total_val_loss / len(val_dataset)
    )

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {average_train_loss:.4f} "
        f"Validation Loss: {average_val_loss:.4f}"
    )

    # Save Best Model


    if average_val_loss < best_val_loss:

        best_val_loss = average_val_loss

        torch.save(
            model.state_dict(),
            "artifacts/scratch_model.pt"
        )


training_time = time.time() - start_time

print("\nTraining completed!")
print("Training time:", round(training_time, 2), "seconds")

**Validation Loss and training graphs**

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    range(1, EPOCHS + 1),
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()
plt.grid()

plt.show()

#15.Evaluate Model on Test set

**Loading Best Model**

In [ ]:
model.load_state_dict(
    torch.load(
        "artifacts/scratch_model.pt",
        map_location=DEVICE
    )
)

model.eval()

print("Best model loaded successfully!")

**Predictions**

In [ ]:
y_true = []
y_pred = []

inference_times = []

with torch.no_grad():

    for input_ids, labels in test_loader:

        input_ids = input_ids.to(DEVICE)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        outputs = model(input_ids)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        inference_times.append(
            elapsed / len(input_ids)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        y_true.extend(labels.tolist())

        y_pred.extend(
            predictions.cpu().tolist()
        )


print("Predictions completed!")

print("Number of test samples:", len(y_true))

**Calculating Evaluation Metrics**

In [ ]:
accuracy = accuracy_score(y_true,y_pred)

macro_f1 = f1_score(y_true,y_pred,average="macro",zero_division=0)

spam_precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

spam_recall = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

spam_f1 = f1_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

average_latency = np.mean(inference_times)

print("Evaluation Results")
print("----------------------------")

print(f"Accuracy: {accuracy:.4f}")

print(f"Macro F1: {macro_f1:.4f}")

print(f"Spam Precision: {spam_precision:.4f}")

print(f"Spam Recall: {spam_recall:.4f}")

print(f"Spam F1: {spam_f1:.4f}")

print(
    f"Average latency per message: "
    f"{average_latency * 1000:.3f} ms"
)

**Classifiction Report**

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Ham", "Spam"],
        zero_division=0
    )
)

**Confusion Matrix**

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.title("Scratch Transformer - Confusion Matrix")

plt.show()

#15.Trying Model on new message

**Prediction function**

In [ ]:
def predict_message(message):

    model.eval()

    encoded = encode_text(
        message,
        vocab,
        MAX_LEN
    )

    input_ids = torch.tensor(
        [encoded],
        dtype=torch.long
    ).to(DEVICE)

    with torch.no_grad():

        outputs = model(input_ids)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predicted_class = torch.argmax(
            probabilities,
            dim=1
        ).item()

    label = ID2LABEL[predicted_class]

    confidence = probabilities[
        0, predicted_class
    ].item()

    return label, confidence


ID2LABEL = {
    0: "Ham",
    1: "Spam"
}

**Testing Message**

In [ ]:
test_messages = [

    "Congratulations! You won a free prize. Call now!",

    "Hey, are we still meeting today?",

    "URGENT! You have won a cash reward!",

    "Can you send me the homework please?"

]

for message in test_messages:

    prediction, confidence = predict_message(
        message
    )

    print("Message:", message)

    print("Prediction:", prediction)

    print(
        "Confidence:",
        round(confidence * 100, 2),
        "%"
    )

    print("-" * 60)

**Save Vocabulary and results**

In [ ]:
# Save vocabulary
with open(
    "artifacts/vocab.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        vocab,
        file,
        ensure_ascii=False,
        indent=2
    )


# Count model parameters
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


# Prepare results
results = {

    "model_name": "Scratch Transformer",

    "dataset_rows": len(df),

    "train_size": len(train_df),

    "validation_size": len(val_df),

    "test_size": len(test_df),

    "seed": SEED,

    "max_length": MAX_LEN,

    "epochs": EPOCHS,

    "batch_size": BATCH_SIZE,

    "parameters": int(total_parameters),

    "training_time_seconds": float(training_time),

    "average_latency_ms": float(
        average_latency * 1000
    ),

    "accuracy": float(accuracy),

    "macro_f1": float(macro_f1),

    "spam_precision": float(spam_precision),

    "spam_recall": float(spam_recall),

    "spam_f1": float(spam_f1),

    "confusion_matrix": cm.tolist()

}


with open(
    "artifacts/scratch_results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        results,
        file,
        indent=4
    )


# Save test predictions
test_predictions = test_df.copy()

test_predictions["predicted_id"] = y_pred

test_predictions["predicted_label"] = [
    ID2LABEL[x] for x in y_pred
]

test_predictions.to_csv(
    "artifacts/scratch_test_predictions.csv",
    index=False
)


print("All artifacts saved successfully!")

# PART 2 – Pretrained DistilBERT + Comparison (Member 2)

## P2 – 1. Install & Libraries

In [ ]:
!pip -q install transformers
import os, json, time, random, zipfile, shutil, platform
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
print("torch", torch.__version__, "| transformers", transformers.__version__)

## P2 – 2. Seed, Device & Hardware (must be documented)

In [ ]:
SEED = 42            # SAME seed as Member 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HARDWARE = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor() or "CPU"
print("Device:", DEVICE, "|", HARDWARE)

## P2 – 3. Load the SAME shared data (upload zips in Colab)

In [ ]:
# Data & Member 1's results come from Part 1 (shared_data/ and artifacts/ folders already created above)
train_df = pd.read_csv("shared_data/train.csv")
val_df   = pd.read_csv("shared_data/validation.csv")
test_df  = pd.read_csv("shared_data/test.csv")
print(len(train_df), len(val_df), len(test_df))
print(test_df["label"].value_counts())
train_df.head()

**Input text choice:** DistilBERT was pretrained on natural, cased-normalised text with punctuation/numbers, so we feed it the **original `message`** column (its own WordPiece tokenizer handles the rest). The *rows* (train/val/test split) are identical to Member 1's – only the text column differs because each model uses its own tokenizer. Set `TEXT_COL = "clean_message"` to use exactly the same cleaned text as the scratch model.

In [ ]:
TEXT_COL = "message"      # or "clean_message"
LABEL_COL = "label_id"    # ham=0, spam=1 (same mapping as Member 1)
ID2LABEL = {0: "Ham", 1: "Spam"}

SMOKE_TEST = False        # True = tiny run just to check the code works
if SMOKE_TEST:
    train_df, val_df, test_df = train_df.head(64), val_df.head(32), test_df.head(32)

## P2 – 4. Pretrained checkpoint & its tokenizer

In [ ]:
CHECKPOINT = "distilbert-base-uncased"   # Hugging Face: https://huggingface.co/distilbert-base-uncased (Apache-2.0)
MAX_LEN    = 128                          # SMS are short; 128 WordPiece tokens is enough
tokenizer  = AutoTokenizer.from_pretrained(CHECKPOINT)

s = train_df[TEXT_COL].iloc[0]
print(s); print(tokenizer.tokenize(s)); print(tokenizer(s)["input_ids"])

## P2 – 5. Dataset & DataLoaders

In [ ]:
class HFSpamDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df[TEXT_COL].astype(str).tolist(), truncation=True, max_length=MAX_LEN)
        self.ids, self.mask = enc["input_ids"], enc["attention_mask"]
        self.labels = df[LABEL_COL].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "attention_mask": self.mask[i]}, self.labels[i]

def collate(batch):
    enc = tokenizer.pad([b[0] for b in batch], return_tensors="pt")   # dynamic padding
    return enc, torch.tensor([b[1] for b in batch], dtype=torch.long)

BATCH_SIZE = 32   # same as Member 1 (also keeps latency measurement comparable)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(HFSpamDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate, generator=g)
val_loader   = DataLoader(HFSpamDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(HFSpamDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
print(len(train_loader), len(val_loader), len(test_loader))

## P2 – 6. Model, loss (same class weights as Member 1), optimizer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=2, id2label=ID2LABEL, label2id={"Ham": 0, "Spam": 1}).to(DEVICE)
total_parameters = sum(p.numel() for p in model.parameters())
print("Parameters:", f"{total_parameters:,}")

counts = train_df[LABEL_COL].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (2 * counts), dtype=torch.float32).to(DEVICE)
print("Class weights:", class_weights)           # should be [0.5723, 3.9598] like Member 1
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS, LR, WEIGHT_DECAY, WARMUP = 3, 2e-5, 0.01, 0.1
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * total_steps), total_steps)

## P2 – 7. Fine-tuning (best epoch picked on VALIDATION loss – never on test)

In [ ]:
os.makedirs("artifacts_distilbert", exist_ok=True)
MODEL_DIR = "artifacts_distilbert/distilbert_model"
best_val_loss, train_losses, val_losses = float("inf"), [], []
start_time = time.time()

for epoch in range(EPOCHS):
    model.train(); total = 0
    for enc, labels in train_loader:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}; labels = labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(**enc).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total += loss.item()
    tr = total / len(train_loader)

    model.eval(); total = 0
    with torch.no_grad():
        for enc, labels in val_loader:
            enc = {k: v.to(DEVICE) for k, v in enc.items()}; labels = labels.to(DEVICE)
            total += criterion(model(**enc).logits, labels).item() * len(labels)
    va = total / len(val_loader.dataset)
    train_losses.append(tr); val_losses.append(va)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {tr:.4f} Validation Loss: {va:.4f}")

    if va < best_val_loss:
        best_val_loss = va
        model.save_pretrained(MODEL_DIR); tokenizer.save_pretrained(MODEL_DIR)

if DEVICE.type == "cuda": torch.cuda.synchronize()
training_time = time.time() - start_time
print("\nFine-tuning time:", round(training_time, 2), "seconds")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,EPOCHS+1), train_losses, marker="o", label="Training Loss")
plt.plot(range(1,EPOCHS+1), val_losses, marker="o", label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("DistilBERT – Training vs Validation Loss")
plt.legend(); plt.grid(); plt.savefig("artifacts_distilbert/distilbert_loss_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## P2 – 8. Evaluate on the held-out TEST set (best checkpoint)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()

y_true, y_pred, y_prob, inference_times = [], [], [], []
with torch.no_grad():
    for enc, labels in test_loader:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = model(**enc).logits
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        inference_times.append((time.perf_counter() - t0) / len(labels))   # same method as Member 1
        y_true += labels.tolist()
        y_pred += logits.argmax(1).cpu().tolist()
        y_prob += torch.softmax(logits, 1)[:, 1].cpu().tolist()

accuracy       = accuracy_score(y_true, y_pred)
macro_f1       = f1_score(y_true, y_pred, average="macro", zero_division=0)
spam_precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
spam_recall    = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
spam_f1        = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
average_latency = float(np.mean(inference_times))

print(f"Accuracy: {accuracy:.4f}\nMacro F1: {macro_f1:.4f}\nSpam Precision: {spam_precision:.4f}")
print(f"Spam Recall: {spam_recall:.4f}\nSpam F1: {spam_f1:.4f}\nAvg latency/message (batch=32): {average_latency*1000:.3f} ms")
print(classification_report(y_true, y_pred, labels=[0,1], target_names=["Ham","Spam"], zero_division=0))

In [ ]:
# Single-message latency (what the Streamlit app will show): tokenization + forward, batch size 1
def predict_message(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): probs = torch.softmax(model(**enc).logits, 1)[0]
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    lat = time.perf_counter() - t0
    c = int(probs.argmax()); return ID2LABEL[c], float(probs[c]), lat

for _ in range(5): predict_message("warm up")           # warm-up
lats = [predict_message(t)[2] for t in test_df[TEXT_COL].tolist()[:100]]
single_latency_ms = float(np.mean(lats) * 1000)
print(f"Single-message latency (incl. tokenization): {single_latency_ms:.2f} ms")

for m in ["Congratulations! You won a free prize. Call now!", "Hey, are we still meeting today?",
          "URGENT! You have won a cash reward!", "Can you send me the homework please?"]:
    l, c, t = predict_message(m); print(f"{m}\n  -> {l} ({c*100:.2f}%)  {t*1000:.1f} ms\n")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Ham","Spam"], yticklabels=["Ham","Spam"])
plt.xlabel("Predicted Label"); plt.ylabel("True Label"); plt.title("DistilBERT – Confusion Matrix")
plt.savefig("artifacts_distilbert/distilbert_confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

## P2 – 9. Save results (same JSON format as Member 1, for Member 3's app)

In [ ]:
results = {
    "model_name": "DistilBERT (fine-tuned)", "checkpoint": CHECKPOINT,
    "dataset_rows": len(train_df)+len(val_df)+len(test_df),
    "train_size": len(train_df), "validation_size": len(val_df), "test_size": len(test_df),
    "seed": SEED, "max_length": MAX_LEN, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LR, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP, "text_column": TEXT_COL,
    "hardware": HARDWARE, "parameters": int(total_parameters),
    "training_time_seconds": float(training_time),
    "average_latency_ms": average_latency*1000,            # batch-of-32 method, same as scratch model
    "single_message_latency_ms": single_latency_ms,        # batch=1 incl. tokenization
    "accuracy": float(accuracy), "macro_f1": float(macro_f1),
    "spam_precision": float(spam_precision), "spam_recall": float(spam_recall), "spam_f1": float(spam_f1),
    "confusion_matrix": cm.tolist(),
}
json.dump(results, open("artifacts_distilbert/distilbert_results.json","w"), indent=4)

preds = test_df.copy()
preds["predicted_id"] = y_pred; preds["predicted_label"] = [ID2LABEL[x] for x in y_pred]
preds["spam_probability"] = y_prob
preds.to_csv("artifacts_distilbert/distilbert_test_predictions.csv", index=False)
print("saved")

## P2 – 10. Compare the two models (same test items)

In [ ]:
scratch = json.load(open("artifacts/scratch_results.json"))
dist    = json.load(open("artifacts_distilbert/distilbert_results.json"))

# sanity check: both models were evaluated on identical test rows
sp = pd.read_csv("artifacts/scratch_test_predictions.csv")
assert len(sp) == len(preds) and (sp["message"].values == preds["message"].values).all(), "Test sets differ!"
assert (sp["label_id"].values == preds["label_id"].values).all()
print("OK: both models evaluated on the same", len(sp), "test messages")

rows = [("Accuracy","accuracy"),("Macro F1","macro_f1"),("Spam Precision","spam_precision"),
        ("Spam Recall","spam_recall"),("Spam F1","spam_f1"),("Parameters","parameters"),
        ("Training time (s)","training_time_seconds"),("Latency / msg, batch=32 (ms)","average_latency_ms")]
comp = pd.DataFrame({"Metric":[r[0] for r in rows],
                     "Scratch Transformer":[scratch[r[1]] for r in rows],
                     "DistilBERT (fine-tuned)":[dist[r[1]] for r in rows]})
comp.to_csv("artifacts_distilbert/model_comparison.csv", index=False)
comp.style.format({"Scratch Transformer":"{:,.4f}","DistilBERT (fine-tuned)":"{:,.4f}"})

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18,5))
metrics = ["accuracy","macro_f1","spam_precision","spam_recall","spam_f1"]
x = np.arange(len(metrics)); w = 0.38
ax[0].bar(x-w/2,[scratch[m] for m in metrics],w,label="Scratch"); ax[0].bar(x+w/2,[dist[m] for m in metrics],w,label="DistilBERT")
ax[0].set_xticks(x); ax[0].set_xticklabels(["Acc","Macro F1","Spam P","Spam R","Spam F1"]); ax[0].set_ylim(0,1.05); ax[0].set_title("Quality metrics"); ax[0].legend()
for a,(r,n) in zip(ax[1:],[(scratch,"Scratch Transformer"),(dist,"DistilBERT")]):
    sns.heatmap(np.array(r["confusion_matrix"]),annot=True,fmt="d",cmap="Blues",ax=a,xticklabels=["Ham","Spam"],yticklabels=["Ham","Spam"])
    a.set_title(n+" – Confusion Matrix"); a.set_xlabel("Predicted"); a.set_ylabel("True")
plt.tight_layout(); plt.savefig("artifacts_distilbert/model_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15,4))
for a,(k,t) in zip(ax,[("parameters","Parameters"),("training_time_seconds","Training time (s)"),("average_latency_ms","Latency / msg (ms)")]):
    a.bar(["Scratch","DistilBERT"],[scratch[k],dist[k]],color=["#4C78A8","#F58518"]); a.set_title(t)
plt.tight_layout(); plt.savefig("artifacts_distilbert/model_cost_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Examples where the models disagree (good for the report)
preds["scratch_pred"] = sp["predicted_label"].values
d = preds[preds["predicted_label"] != preds["scratch_pred"]][["message","label","scratch_pred","predicted_label"]]
d.columns = ["message","true","scratch","distilbert"]
print("Disagreements:", len(d)); d.head(10)

## P2 – 11. Explanation for the report
*(Fill this in after you see YOUR measured numbers – the brief requires measured values only.)*

- **Performance difference:** …
- **Why pretraining helps:** DistilBERT already learned grammar/word meaning from large text corpora, so fine-tuning only adapts it to spam; the scratch model must learn everything from ~4,100 SMS.
- **Cost trade-off:** DistilBERT has ~66M parameters vs ~1.15M for the scratch model → slower inference and larger file, but usually better quality.
- **Limitations:** small dataset, one random seed, class imbalance (12.6% spam), latency depends on hardware (CPU vs GPU).

## P2 – 12. Package artifacts for Member 3

In [ ]:
# One zip with everything Member 3 needs
os.makedirs("project_artifacts", exist_ok=True)
for d in ["artifacts", "artifacts_distilbert", "shared_data"]:
    shutil.copytree(d, f"project_artifacts/{d}", dirs_exist_ok=True)
shutil.make_archive("project_artifacts", "zip", "project_artifacts")
from google.colab import files
files.download("project_artifacts.zip")